In [1]:
import json
import re

# Input/output files
input_file = "../tune_fix.txt"
output_file = "tune_fix.json"

# Read the entire log file
with open(input_file, "r", encoding="utf-8") as f:
    text = f.read()


# Split the log into epoch blocks
blocks = re.split(r"=+\s*Epoch\s+(\d+)\s*=+", text)

epochs = []

# blocks looks like:
# [text_before_first_epoch, epoch_number, epoch_text, epoch_number, epoch_text, ...]
for i in range(1, len(blocks), 2):
    epoch = int(blocks[i])
    block = blocks[i + 1]

    data = {"epoch": epoch}

    # Train loss
    match = re.search(r"Train loss:\s*([-+]?\d*\.?\d+)", block)
    if match:
        data["train_loss"] = float(match.group(1))

    # NLL on validation set
    match = re.search(r"NLL on val set:\s*([-+]?\d*\.?\d+)", block)
    if match:
        data["val_nll"] = float(match.group(1))

    # Posterior NLL on validation set
    match = re.search(r"Posterior NLL on val:\s*([-+]?\d*\.?\d+)", block)
    if match:
        data["posterior_val_nll"] = float(match.group(1))

    # Posterior NLL on training set
    match = re.search(r"Posterior NLL on train:\s*([-+]?\d*\.?\d+)", block)
    if match:
        data["posterior_train_nll"] = float(match.group(1))

    # Prior NLL on training set
    match = re.search(r"Prior NLL on train:\s*([-+]?\d*\.?\d+)", block)
    if match:
        data["prior_train_nll"] = float(match.group(1))

    # Layered KL
    match = re.search(r"Layered KL:\s*tensor\(\[([^\]]+)\]", block)
    if match:
        data["layered_kl"] = [
            float(x.strip())
            for x in match.group(1).split(",")
        ]

    # KL target
    match = re.search(r"KL target set to\s*([-+]?\d*\.?\d+)", block)
    if match:
        data["kl_target"] = float(match.group(1))

    # Outer step
    match = re.search(
        r"Outer step:\s*"
        r"expected KL=([-+]?\d*\.?\d+),\s*"
        r"residual=([-+]?\d*\.?\d+),\s*"
        r"lambda=([-+]?\d*\.?\d+),\s*"
        r"rho=([-+]?\d*\.?\d+),\s"
        r"Expected Wasserstein=([-+]?\d*\.?\d+),\s"
        r"latent mean diff=([-+]?\d*\.?\d+),\s"
        r"latent logvar diff=([-+]?\d*\.?\d+)",
        block,
    )

    if match:
        data["expected_kl"] = float(match.group(1))
        data["residual"] = float(match.group(2))
        data["lambda"] = float(match.group(3))
        data["rho"] = float(match.group(4))
        data["wasserstein"] = float(match.group(5))
        data["mean_diff"] = float(match.group(6))
        data["logvar_diff"] = float(match.group(7))
        

    epochs.append(data)


# Write parsed data to JSON
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(epochs, f, indent=4)

print(f"Parsed {len(epochs)} epochs")
print(f"Saved to {output_file}")

Parsed 128 epochs
Saved to tune_fix.json
